# Data Science Internship Assessment

### Problem Framing
The company requires high-quality, reliable agricultural data to drive its core crop recommendation engine. The system must synthesize complex agricultural factors—including soil chemical properties, dynamic climate conditions, geographical locations, and historical crop performance—to deliver precise, actionable insights to farmers. This notebook establishes an end-to-end data pipeline that sources, cleans, analyzes, and prepares a baseline crop recommendation dataset. The objective is to evaluate data integrity, extract clear agronomic insights, engineer robust features, and build an interpretable recommendation logic framework that scales efficiently within the companies operational ecosystem.

## Phase 1: Setup & Task 1: Data Sourcing

In [3]:
import numpy as np
import pandas as pd
import missingno as msno 
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import statsmodels.graphics.correlation as sgc
from statsmodels.graphics.gofplots import qqplot
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import OLSInfluence
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

### Task 1: Data Sourcing

#### Data Provenance and Profile

1. **Source:** The dataset used is the widely cited "Crop Recommendation Dataset" compiled by Atharva Ingle, hosted on [Kaggle](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset). It is an open-source dataset built by blending real-world agricultural metrics to form a clean benchmark for recommendation algorithms.

2. **Description:** The dataset consists of 2,200 observations distributed across 8 columns. It contains 7 numeric features—Nitrogen (N), Phosphorus (P), and Potassium (K) soil values measured in ppm; Temperature (°C); Relative Humidity (%); pH; and Rainfall (mm)—alongside 1 categorical target column (`label`). The target contains 22 unique crop classes (e.g., rice, maize, mango), perfectly balanced with exactly 100 rows per class.

3. **Rationale for Selection:** This dataset offers a direct 1:1 alignment with three of Rhea's primary recommendation dimensions: **soil properties** (N, P, K, pH) and **climate conditions**
(temperature, humidity, rainfall), mapped against **crop performance** (the successful crop label). \
*Limitation Note:* Specific geospatial/location identifiers are absent, serving as a primary structural constraint to address when moving this logic to production.

4. **Value to Rhea:** By providing a clean, multi-class labeled ground truth mapping environmental inputs to viable crops, this dataset allows us to build and validate baseline similarity matchers. This establishes a clear performance benchmark for Rhea's recommendation logic before incorporating more complex field data.

In [5]:
# Load the Raw Dataset
data_path = "/home/kobey/Documents/DATASCIENCE/Rhea Data Science/data/raw/Crop_recommendation.csv"
df = pd.read_csv(data_path)

# Display dataset dimensions, structural metadata, and initial rows
print(f"Dataset Shape: {df.shape}\n")
print("--- Dataset Info ---")
df.info()
print("\n--- First 5 Rows ---")
df.head()


Dataset Shape: (2200, 8)

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   object 
dtypes: float64(4), int64(3), object(1)
memory usage: 137.6+ KB

--- First 5 Rows ---


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


### Initial Data Interpretation
The structural inspection confirms that the dataset contains exactly 2,200 rows and 8 columns. The feature data types align perfectly with our requirements: the primary soil macronutrients (`N`, `P`, `K`) are natively stored as integers, while the environmental conditions (`temperature`, `humidity`, `ph`, `rainfall`) are represented as floating-point decimals. The target variable (`label`) is loaded as an object type, which we will convert to a categorical data type during the cleaning phase. No initial structure deviations or anomalies are apparent from this high-level look.